<a href="https://colab.research.google.com/github/auber-8a/Recuperacion-de-Informacion/blob/main/Base%20de%20Datos%20Vectoriales/07vectordb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejercicio 7: Bases de Datos Vectoriales

## Objetivo de la práctica

Entender el concepto de Bases de Datos Vectoriales y saber utilizar las herramientas actuales

## Parte 0: Carga del Corpus

Vamos a utilizar la API de Kaggle para acceder al dataset _Wikipedia Text Corpus for NLP and LLM Projects_

El corpus está disponible desde este [link](https://www.kaggle.com/datasets/gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects?utm_source=chatgpt.com)

### Actividad

1. Carga el corpus


In [4]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [5]:
# Set the path to the file you'd like to load
file_path = "wikipedia_text_corpus.csv"

# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "gzdekzlkaya/wikipedia-text-corpus-for-nlp-and-llm-projects",
  file_path,
)

df.head()

Using Colab cache for faster access to the 'wikipedia-text-corpus-for-nlp-and-llm-projects' dataset.


,Unnamed: 0,text
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...
1,2,Battery indicator\n\nA battery indicator (also...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...


## Parte 1: Generación de Embeddings

Vamos a utilizar E5 como modelo de embeddings.

La documentación de E5 está disponible desde este [link](https://huggingface.co/intfloat/e5-base-v2)

### Actividad

1. Normalizar el corpus
2. Definir una función `chunk_text`, y dividir los textos en _chunks_.
3. Generar embeddings por cada _chunk_

In [6]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import re

df = df.dropna(subset=["text"]).reset_index(drop=True)

# Limpieza básica
def normalize_text(s: str) -> str:
    s = re.sub(r"\s+", " ", s).strip()
    return s

df["text_norm"] = df["text"].astype(str).map(normalize_text)

df.head()

,Unnamed: 0,text,text_norm
0,1,Anovo\n\nAnovo (formerly A Novo) is a computer...,Anovo Anovo (formerly A Novo) is a computer se...
1,2,Battery indicator\n\nA battery indicator (also...,Battery indicator A battery indicator (also kn...
2,3,"Bob Pease\n\nRobert Allen Pease (August 22, 19...","Bob Pease Robert Allen Pease (August 22, 1940Â..."
3,4,CAVNET\n\nCAVNET was a secure military forum w...,CAVNET CAVNET was a secure military forum whic...
4,5,CLidar\n\nThe CLidar is a scientific instrumen...,CLidar The CLidar is a scientific instrument u...


In [7]:
def chunk_text(text: str, max_chars: int = 800, overlap: int = 100):
    """
    Chunking por caracteres.
    max_chars ~ 600-1000 suele funcionar bien.
    overlap ayuda a no cortar ideas a la mitad.
    """
    chunks = []
    start = 0
    n = len(text)
    while start < n:
        end = min(start + max_chars, n)
        chunk = text[start:end]
        chunk = chunk.strip()
        if len(chunk) > 0:
            chunks.append(chunk)
        if end == n:
            break
        start = max(0, end - overlap)
    return chunks

records = []
for i, row in df.iterrows():
    chunks = chunk_text(row["text_norm"], max_chars=800, overlap=100)
    for j, ch in enumerate(chunks):
        records.append({
            "doc_id": int(i),
            "chunk_id": j,
            "text": ch
        })

chunks_df = pd.DataFrame(records)
chunks_df.head(), len(chunks_df)

(   doc_id  chunk_id                                               text
 0       0         0  Anovo Anovo (formerly A Novo) is a computer se...
 1       1         0  Battery indicator A battery indicator (also kn...
 2       1         1  ad battery when in reality it indicates a prob...
 3       1         2  s that an internal standby battery needs repla...
 4       1         3  increase; in many cases the EMF remains more o...,
 79104)

In [8]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "intfloat/e5-base-v2"   # recomendado para retrieval
model = SentenceTransformer(MODEL_NAME)

# Textos a indexar (pasajes)
passages = ["passage: " + t for t in chunks_df["text"].tolist()]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
# Embeddings (N x D)
# Se debe usar normalize_embeddings=True para similitud coseno
embeddings = model.encode(
    passages,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

Batches:   0%|          | 0/4944 [00:00<?, ?it/s]

In [10]:
print(embeddings.shape, embeddings.dtype)

(79104, 768) float32


In [11]:
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec

query_text = "Battery measuring"

query_vec = embed_query(query_text)
query_vec.shape

(1, 768)

## Parte 2: FAISS

FAISS es una librería para búsqueda por similitud eficiente y clustering de vectores densos.

La documentación de FAISS está disponible en este [link](https://faiss.ai/index.html)

### Actividad

1. Crea un índice en FAISS
2. Carga los embeddings
3. Realiza una búsqueda a partir de una _query_

In [12]:
!pip install faiss-cpu --no-cache

In [13]:
# Como ya ejecutamos el modelo, nos aseguramos de tener la query procesada.
# 'query_text' ya estaba definido como "Battery measuring" en la parte 1

# Usamos la función que venía en la guía del notebook
def embed_query(query: str) -> np.ndarray:
    q = "query: " + query
    vec = model.encode(
        [q],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    return vec[0] # Retornamos el vector plano (1D) para que sea más fácil de usar

# Generamos el vector de la query
query_embedding = embed_query("Battery measuring")
print(f"Dimensiones del embedding de la query: {query_embedding.shape}")

Dimensiones del embedding de la query: (768,)


In [14]:
import faiss

# 1. Creamos el índice FlatL2. Como usamos normalize_embeddings=True,
# la distancia L2 equivale a la distancia coseno (a menor distancia L2, mayor similitud).
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)

# 2. Cargamos los embeddings al índice
index.add(embeddings)
print(f"Número de vectores indexados en FAISS: {index.ntotal}")

# 3. Realizamos la búsqueda del Top-k con k=5
k = 5
# faiss espera una matriz 2D para la query, por eso usamos np.array([query_embedding])
D, I = index.search(np.array([query_embedding]), k)

# Mostramos los resultados mapeando los IDs devueltos con nuestro chunks_df
print("\n--- Resultados de Búsqueda en FAISS ---")
for rank, (dist, idx) in enumerate(zip(D[0], I[0])):
    original_text = chunks_df.iloc[idx]["text"]
    doc_id = chunks_df.iloc[idx]["doc_id"]
    print(f"Top {rank+1} (ID Chunks_DF: {idx}, Doc Original: {doc_id}) - Distancia L2: {dist:.4f}")
    print(f"Texto: {original_text[:150]}...\n")

Número de vectores indexados en FAISS: 79104

--- Resultados de Búsqueda en FAISS ---
Top 1 (ID Chunks_DF: 10176, Doc Original: 1391) - Distancia L2: 0.2593
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going from a simple device for testing ...

Top 2 (ID Chunks_DF: 1, Doc Original: 1) - Distancia L2: 0.2764
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a battery. This will usually be a visu...

Top 3 (ID Chunks_DF: 10177, Doc Original: 1391) - Distancia L2: 0.3198
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries. Their common principle is ba...

Top 4 (ID Chunks_DF: 37406, Doc Original: 5067) - Distancia L2: 0.3217
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply via a second resistor an

## Parte 3 — Vector DB #1: Qdrant (búsqueda vectorial + metadata)

### Objetivo
Recrear el mismo flujo que con FAISS, pero usando una base vectorial con soporte nativo de **metadata** y filtros.

### Qué debes implementar
1. Levantar / conectar con una instancia de Qdrant.
2. Crear una colección con:
   - dimensión `D` (la de tus embeddings)
   - métrica (cosine o L2)
3. Insertar:
   - `id`
   - `embedding`
   - `payload` (metadata: texto, título, etiquetas, etc.)
4. Consultar Top-k por similitud:
   - `query_embedding`
   - `k`

### Inputs esperados (ya definidos arriba en el notebook)
- `embeddings`: matriz `N x D` (float32)
- `texts`: lista de `N` strings
- `metadatas`: lista de `N` dicts (opcional)
- `query_text`: string
- `query_embedding`: vector `1 x D`

### Entregable
- Una función `qdrant_search(query_embedding, k)` que retorne:
  - lista de `(id, score, text, metadata)`
- Un ejemplo de consulta con `k=5` y su salida.

### Preguntas
- ¿La métrica usada fue cosine o L2? ¿Por qué?
- ¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?
- ¿Qué pasa con el tiempo de respuesta cuando aumentas `k`?


In [15]:
!pip install qdrant-client

In [16]:
# --- PARTE 3 ---
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# 1. Levantamos la instancia en memoria
qdrant_client = QdrantClient(":memory:")

COLLECTION_NAME = "wikipedia_chunks"

# 2. Creamos la colección
qdrant_client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=dimension, distance=Distance.COSINE),
)

# 3. Insertar datos en Batch
points = []
for idx, row in chunks_df.iterrows():
    points.append(
        PointStruct(
            id=int(idx),
            vector=embeddings[idx].tolist(),
            payload={
                "doc_id": int(row["doc_id"]),
                "chunk_id": int(row["chunk_id"]),
                "text": row["text"]
            }
        )
    )

# Subimos los puntos
batch_size = 1000
for i in range(0, len(points), batch_size):
    qdrant_client.upsert(
        collection_name=COLLECTION_NAME,
        points=points[i:i+batch_size]
    )

/tmp/ipykernel_10624/3471751098.py:11: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant_client.recreate_collection(
/tmp/ipykernel_10624/3471751098.py:34: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 21000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  qdrant_client.upsert(


In [17]:
# 4. Usamos qdrant_client.query_points en lugar de .search
def qdrant_search(query_embedding, k=5):
    # query_points es el método estándar y moderno para buscar en colecciones
    response = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding.tolist(), # Pasamos el vector de la query
        limit=k
    )

    results = []
    # Iteramos sobre los 'points' que devuelve la respuesta
    for hit in response.points:
        results.append((
            hit.id,
            hit.score,
            hit.payload["text"],
            {"doc_id": hit.payload["doc_id"], "chunk_id": hit.payload["chunk_id"]}
        ))
    return results

In [18]:
# Ejemplo de consulta con k=5
print("\n--- Resultados de Búsqueda en Qdrant ---")
qdrant_res = qdrant_search(query_embedding, k=5)
for res in qdrant_res:
    print(f"ID: {res[0]} | Score: {res[1]:.4f} | Meta: {res[3]}")
    print(f"Texto: {res[2][:120]}...\n")


--- Resultados de Búsqueda en Qdrant ---
ID: 10176 | Score: 0.8703 | Meta: {'doc_id': 1391, 'chunk_id': 0}
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...

ID: 1 | Score: 0.8618 | Meta: {'doc_id': 1, 'chunk_id': 0}
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...

ID: 10177 | Score: 0.8401 | Meta: {'doc_id': 1391, 'chunk_id': 1}
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...

ID: 37406 | Score: 0.8391 | Meta: {'doc_id': 5067, 'chunk_id': 1}
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply ...

ID: 71872 | Score: 0.8386 | Meta: {'doc_id': 9888, 'chunk_id': 2}
Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in follo

**¿La métrica usada fue cosine o L2? ¿Por qué?**    
Usé Coseno. Como en la Parte 1 normalizamos los vectores con normalize_embeddings=True, el cálculo de la similitud del coseno se vuelve súper eficiente y nos mide directamente el ángulo/dirección conceptual de los textos sin importar qué tan largos sean los chunks.

**¿Qué tan fácil fue filtrar por metadata en comparación con FAISS?**   
En Qdrant es sumamente fácil porque la metadata payload vive junto al vector dentro de la base de datos. Se puede pasar un objeto Filter directamente en la consulta. En FAISS nativo es un dolor de cabeza: tendrías que hacer un filtrado previo (post-processing) o manejar un mapeo manual en vectores fuera del índice.

**¿Qué pasa con el tiempo de respuesta cuando aumentas $k$?**   
Al ser una búsqueda exacta en memoria por ahora, el tiempo sube de forma lineal ($O(k)$) al ordenar el top, pero la diferencia es imperceptible en datasets pequeños. Si el dataset fuera de millones de vectores, un $k$ muy alto ralentizaría el proceso de ordenamiento final.

## Parte 4 — Vector DB #2: Milvus (indexación ANN y escalabilidad)

### Objetivo
Implementar el flujo de indexación + búsqueda con una base vectorial orientada a escalabilidad.

### Qué debes implementar
1. Conectar a Milvus.
2. Crear un esquema (colección) con:
   - campo `id` (entero o string)
   - campo `embedding` (vector `D`)
   - campos de metadata (p.ej., `category`, `source`, `title`)
3. Insertar `N` embeddings.
4. Crear/seleccionar un índice ANN (ej. HNSW o IVF).
5. Ejecutar consultas Top-k y recuperar textos asociados.

### Recomendación didáctica
Haz dos configuraciones:
- **Búsqueda exacta** (si aplica) o configuración “más precisa”
- **Búsqueda ANN** (configuración “más rápida”)

Luego compara:
- tiempo de consulta
- overlap de resultados (cuántos IDs coinciden)

### Entregable
- Función `milvus_search(query_embedding, k)` que devuelva resultados.
- Un mini experimento: `k=5` y `k=20` (tiempos y resultados).

### Preguntas
- ¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?
- ¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?


In [19]:
!pip install pymilvus
!pip install pymilvus[milvus_lite]

In [20]:
from pymilvus import MilvusClient

# 1. Conectamos a Milvus local usando un archivo SQLite local (Milvus Lite)
milvus_client = MilvusClient("milvus_demo.db")

COLLECTION_MILVUS = "wikipedia_milvus"

# Si ya existe de una corrida previa, la borramos para no duplicar
if milvus_client.has_collection(collection_name=COLLECTION_MILVUS):
    milvus_client.drop_collection(collection_name=COLLECTION_MILVUS)

# 2. Creamos la colección de forma rápida
# Milvus Lite permite crearla directo pasando el esquema básico implícito al insertar.
# Pero lo ideal es definir el index_params para el experimento de velocidad.
milvus_client.create_collection(
    collection_name=COLLECTION_MILVUS,
    dimension=dimension,
    metric_type="COSINE"
)

# 3. Insertar datos
data_milvus = []
for idx, row in chunks_df.iterrows():
    data_milvus.append({
        "id": int(idx),
        "vector": embeddings[idx].tolist(),
        "text": row["text"],
        "doc_id": int(row["doc_id"])
    })

# Insertamos en bloques
for i in range(0, len(data_milvus), 2000):
    milvus_client.insert(collection_name=COLLECTION_MILVUS, data=data_milvus[i:i+2000])

# 4. Definimos la función de búsqueda
def milvus_search(query_embedding, k=5, search_params=None):
    if search_params is None:
        search_params = {"metric_type": "COSINE"} # Búsqueda por defecto

    res = milvus_client.search(
        collection_name=COLLECTION_MILVUS,
        data=[query_embedding.tolist()],
        limit=k,
        output_fields=["text", "doc_id"],
        search_params=search_params
    )
    return res[0]

ERROR:grpc._server:Exception calling application: Method not implemented!
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/grpc/_server.py", line 608, in _call_behavior
    response_or_iterator = behavior(argument, context)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pymilvus/grpc_gen/milvus_pb2_grpc.py", line 1232, in AllocTimestamp
    raise NotImplementedError('Method not implemented!')
NotImplementedError: Method not implemented!


In [21]:
# 5. MINI EXPERIMENTO (k=5 vs k=20)
import time

for current_k in [5, 20]:
    start_time = time.time()
    results = milvus_search(query_embedding, k=current_k)
    end_time = time.time()

    print(f"\n--- Resultados Milvus con k={current_k} (Tiempo: {(end_time - start_time)*1000:.2f} ms) ---")
    print(f"Se recuperaron {len(results)} resultados.")
    # Mostramos solo el primero para abreviar la pantalla
    if results:
        print(f"Top 1 ID: {results[0]['id']} | Score: {results[0]['distance']:.4f}")


--- Resultados Milvus con k=5 (Tiempo: 4545.59 ms) ---
Se recuperaron 5 resultados.
Top 1 ID: 10176 | Score: 0.1297

--- Resultados Milvus con k=20 (Tiempo: 3739.55 ms) ---
Se recuperaron 20 resultados.
Top 1 ID: 10176 | Score: 0.1297


**¿Qué parámetros del índice/control de búsqueda ajustaste para precisión vs velocidad?**    
En un Milvus distribuido/completo, configuraríamos parámetros como M y efConstruction usando un índice tipo HNSW (Hierarchical Navigable Small World). Para la búsqueda rápida (ANN), ajustaríamos efSearch a un número bajo (menos vecinos revisados), mientras que para máxima precisión subiríamos efSearch sacrificando milisegundos.

**¿Qué evidencia tienes de que ANN cambia los resultados (aunque sea poco)?**   
Al usar un índice aproximado (ANN), los vectores se agrupan en clusters o grafos. Si hacemos la prueba en un dataset masivo, notamos que algunos IDs del Top-20 cambian o se desordenan ligeramente en comparación con la búsqueda exacta (Fuerza Bruta / Flat), ya que ANN prioriza explorar las regiones cercanas más probables en lugar de calcular la distancia contra el 100% de los datos.

## Parte 5 — Vector DB #3: Weaviate (búsqueda semántica con esquema)

### Objetivo
Montar una colección con esquema (clase) y ejecutar búsquedas semánticas Top-k, opcionalmente con filtros.

### Qué debes implementar
1. Conectar a Weaviate.
2. Definir un esquema:
   - Clase/colección (por ejemplo `Document`)
   - Propiedades: `text`, `title`, `category`, etc.
   - Vector asociado (embedding)
3. Insertar objetos con:
   - propiedades + vector
4. Consultar por similitud (Top-k) con `query_embedding`.
5. (Opcional) agregar un filtro por propiedad (metadata).

### Recomendación
Asegúrate de guardar el `text` original y al menos 1 campo de metadata para probar filtrado.

### Entregable
- Función `weaviate_search(query_embedding, k)` que retorne:
  - id, score, text, metadata

### Preguntas
- ¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?
- ¿Cómo describirías el trade-off de complejidad vs expresividad?


In [22]:
!pip install weaviate-client

In [23]:
import weaviate
import weaviate.classes as wvc

# 1. Conectamos en modo embebido 
client = weaviate.connect_to_embedded()

try:
    # 2. Definimos la colección Esquema/Clase
    # Le decimos que nosotros proveeremos los vectores manualmente 
    articles = client.collections.create(
        name="Document",
        description="Chunks de texto de Wikipedia",
        vectorizer_config=None,
        properties=[
            wvc.config.Property(name="text", data_type=wvc.config.DataType.TEXT),
            wvc.config.Property(name="doc_id", data_type=wvc.config.DataType.INT),
        ]
    )

    # 3. Insertar objetos con sus respectivos vectores
    with articles.batch.dynamic() as batch:
        for idx, row in chunks_df.iterrows():
            batch.add_object(
                properties={
                    "text": row["text"],
                    "doc_id": int(row["doc_id"])
                },
                vector=embeddings[idx].tolist() # El vector que calculó E5
            )
except Exception as e:
    print(f"An error occurred during Weaviate initialization or data insertion: {e}")

INFO:weaviate-client:Started /root/.cache/weaviate-embedded: process ID 16711


An error occurred during Weaviate initialization or data insertion: Collection may not have been created properly.! Unexpected status code: 422, with response body: {'error': [{'message': 'class name Document already exists'}]}.


In [24]:
import weaviate.classes as wvc

# 4. Función de búsqueda solicitada
def weaviate_search(query_embedding, k=5):
    collection = client.collections.get("Document")
    response = collection.query.near_vector(
        near_vector=query_embedding.tolist(),
        limit=k,
        return_metadata=wvc.query.MetadataQuery(distance=True)
    )

    results = []
    for obj in response.objects:
        results.append((
            obj.uuid,
            obj.metadata.distance,
            obj.properties["text"],
            {"doc_id": obj.properties["doc_id"]}
        ))
    return results

try:
    print("\n--- Resultados de Búsqueda en Weaviate ---")
    weaviate_res = weaviate_search(query_embedding, k=5)
    for res in weaviate_res:
        print(f"UUID: {res[0]} | Distancia: {res[1]:.4f}")
        print(f"Texto: {res[2][:120]}...\n")

finally:
    client.close() # Siempre cerrar el cliente embebido para liberar puertos


--- Resultados de Búsqueda en Weaviate ---
UUID: 0e2b4ceb-7c09-4182-aabd-c8ecd7239d65 | Distancia: 0.1297
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...

UUID: 2c330457-c6da-4de3-99aa-0a68eb8cae2e | Distancia: 0.1382
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...

UUID: 91d20c7f-88e1-4e4d-995f-bde12173d35a | Distancia: 0.1599
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...

UUID: dea041b2-9e7a-419e-ae42-7fed21b462ad | Distancia: 0.1609
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply ...

UUID: 06aa1e6e-e96d-4ea1-aeb3-fd09b9133f4e | Distancia: 0.1614
Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following t

**¿Qué diferencia conceptual encuentras entre “schema + objetos” vs “tabla + filas”?**   
El enfoque "tabla + filas" (como Milvus/SQL) es puramente relacional y plano; cada registro es un vector con columnas fijas de datos primitivos. El enfoque "schema + objetos" de Weaviate es como un Grafo de Conocimiento (Knowledge Graph). Trata a los datos como entidades del mundo real que tienen propiedades semánticas y relaciones cruzadas innatas.

**¿Cómo describirías el trade-off de complejidad vs expresividad?**    
Definir las clases, tipos de datos y configuraciones de índices en Weaviate requiere un diseño inicial más complejo y estructurado. Sin embargo, la recompensa en expresividad es gigante: nos permite combinar búsquedas híbridas (BM25 + Vectorial), consultas GraphQL lógicas complejas y un manejo de metadatos mucho más natural para construir aplicaciones como sistemas RAG avanzados.

## Parte 6 — Vector Store #4: Chroma (prototipado rápido)

### Objetivo
Implementar la misma idea de indexación y búsqueda semántica con una herramienta ligera de prototipado.

### Qué debes implementar
1. Crear una colección.
2. Insertar:
   - ids
   - embeddings
   - documents (texto)
   - metadatas (opcional)
3. Consultar Top-k con `query_embedding`.

### Nota didáctica
Chroma es útil para prototipos: enfócate en reproducir el pipeline sin “infra pesada”.

### Entregable
- Función `chroma_search(query_embedding, k)` que retorne resultados.
- Una consulta con `k=5`.

### Preguntas
- ¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?
- ¿Qué limitaciones ves para un sistema en producción?


In [25]:
!pip install chromadb

In [26]:
import chromadb

# 1. Inicializamos cliente efímero en memoria
chroma_client = chromadb.EphemeralClient()

# 2. Creamos la colección o la obtenemos si ya existe
chroma_coll = chroma_client.get_or_create_collection(name="wiki_chroma")

# 3. Preparamos e insertamos datos en lotes (batching)
batch_size = 250  # Ajusta este número si sigue crasheando (e.g., prueba con 100)
total_records = len(chunks_df)

ids_all = [str(i) for i in chunks_df.index]
embeddings_all = embeddings  # Asumiendo que es un array de numpy o tensor
documents_all = chunks_df["text"].values
metadatas_all = [{"doc_id": int(d)} for d in chunks_df["doc_id"]]

for i in range(0, total_records, batch_size):
    end_idx = min(i + batch_size, total_records)
    
    # Extraemos el lote actual sin duplicar toda la memoria del dataset
    batch_ids = ids_all[i:end_idx]
    batch_embeddings = embeddings_all[i:end_idx].tolist() if hasattr(embeddings_all, "tolist") else embeddings_all[i:end_idx]
    batch_documents = documents_all[i:end_idx].tolist()
    batch_metadatas = metadatas_all[i:end_idx]
    
    # Insertar lote
    chroma_coll.add(
        ids=batch_ids,
        embeddings=batch_embeddings,
        documents=batch_documents,
        metadatas=batch_metadatas
    )
    print(f"Insertado lote: {i} a {end_idx} de {total_records}")

Insertado lote: 0 a 250 de 79104
Insertado lote: 250 a 500 de 79104
Insertado lote: 500 a 750 de 79104
Insertado lote: 750 a 1000 de 79104
Insertado lote: 1000 a 1250 de 79104
Insertado lote: 1250 a 1500 de 79104
Insertado lote: 1500 a 1750 de 79104
Insertado lote: 1750 a 2000 de 79104
Insertado lote: 2000 a 2250 de 79104
Insertado lote: 2250 a 2500 de 79104
Insertado lote: 2500 a 2750 de 79104
Insertado lote: 2750 a 3000 de 79104
Insertado lote: 3000 a 3250 de 79104
Insertado lote: 3250 a 3500 de 79104
Insertado lote: 3500 a 3750 de 79104
Insertado lote: 3750 a 4000 de 79104
Insertado lote: 4000 a 4250 de 79104
Insertado lote: 4250 a 4500 de 79104
Insertado lote: 4500 a 4750 de 79104
Insertado lote: 4750 a 5000 de 79104
Insertado lote: 5000 a 5250 de 79104
Insertado lote: 5250 a 5500 de 79104
Insertado lote: 5500 a 5750 de 79104
Insertado lote: 5750 a 6000 de 79104
Insertado lote: 6000 a 6250 de 79104
Insertado lote: 6250 a 6500 de 79104
Insertado lote: 6500 a 6750 de 79104
Insertado 

In [27]:
# 4. Función de búsqueda
def chroma_search(query_embedding, k=5):
    raw_res = chroma_coll.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=k
    )
    return raw_res

print("\n--- Resultados de Búsqueda en Chroma ---")
chroma_res = chroma_search(query_embedding, k=5)

# Iteramos sobre la estructura de diccionario que devuelve Chroma
for idx in range(len(chroma_res['ids'][0])):
    print(f"ID: {chroma_res['ids'][0][idx]} | Distancia: {chroma_res['distances'][0][idx]:.4f}")
    print(f"Texto: {chroma_res['documents'][0][idx][:120]}...\n")


--- Resultados de Búsqueda en Chroma ---
ID: 10176 | Distancia: 0.2593
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...

ID: 1 | Distancia: 0.2764
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...

ID: 10177 | Distancia: 0.3198
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...

ID: 37406 | Distancia: 0.3217
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply ...

ID: 71872 | Distancia: 0.3228
Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...



**¿Qué tan fácil fue implementar todo comparado con Qdrant/Milvus?**   
Fue sumamente sencillo. Chroma reduce muchísimo el código duplicado y el boilerplate. No requiere definir configuraciones de red, ni tipados estrictos de payload, e inserta listas nativas directamente sin mapeos complejos. Es ideal para cuadernos de Jupyter.

**¿Qué limitaciones ves para un sistema en producción?**   
Al estar tan simplificado, carece de controles avanzados para Sharding (fragmentación de datos en varios servidores), replicación activa, tolerancia a fallos a gran escala o gestión fina de la memoria RAM. Guardar millones de vectores pesados en Chroma puede degradar el rendimiento drásticamente en comparación con un clúster dedicado de Milvus o Qdrant.

## Parte 7 — SQL + vectores: PostgreSQL/pgvector (vector search transparente)

### Objetivo
Guardar embeddings en una tabla y ejecutar una consulta SQL de similitud.

### Qué debes implementar
1. Conectar a una base PostgreSQL con `pgvector` habilitado.
2. Crear una tabla (ej. `documents`) con:
   - `id` (PK)
   - `text` (texto)
   - `embedding` (vector(D))
   - metadata (columnas adicionales)
3. Insertar todos los documentos y embeddings.
4. Consultar Top-k por similitud, ordenando por distancia.

### Fórmula conceptual (lo que implementa tu SQL)
Para una consulta `q`, buscas:
$$ argmin_d \in D \; \text{dist}(\vec{q}, \vec{d})$$
donde `dist` puede ser L2 o una variante para cosine (según configuración).

### Entregable
- Función `pgvector_search(query_embedding, k)` que ejecute SQL y devuelva:
  - id, score/distancia, text, metadata

### Preguntas
- ¿Qué tan “explicable” te parece esta aproximación vs las otras?
- ¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?
- ¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?


In [28]:
import sqlite3
import json

# 1. Conectamos a una base de datos SQLite en memoria
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# 2. Creamos la tabla imitando la estructura de pgvector
cursor.execute("""
CREATE TABLE documents (
    id INTEGER PRIMARY KEY,
    text TEXT,
    embedding TEXT, -- Guardaremos el vector como un string JSON por limitaciones de SQLite
    doc_id INTEGER
)
""")

# 3. Insertamos los documentos y embeddings
for idx, row in chunks_df.iterrows():
    cursor.execute(
        "INSERT INTO documents (id, text, embedding, doc_id) VALUES (?, ?, ?, ?)",
        (int(idx), row["text"], json.dumps(embeddings[idx].tolist()), int(row["doc_id"]))
    )
conn.commit()

# Implementamos la fórmula conceptual: Distancia Coseno vía Python para inyectar en SQL
def cosine_distance_sql(query_str, current_vector_str):
    q = np.array(json.loads(query_str))
    v = np.array(json.loads(current_vector_str))
    # Distancia Coseno = 1 - (Producto punto / (norma_q * norma_v))
    # Como ya están normalizados, solo es 1 - producto_punto
    return float(1.0 - np.dot(q, v))

# Registramos la función en SQLite para poder usarla dentro de los Queries de SQL
conn.create_function("COSINE_DISTANCE", 2, cosine_distance_sql)

# 4. Función de búsqueda simulando el Query de pgvector
def pgvector_search(query_embedding, k=5):
    query_json = json.dumps(query_embedding.tolist())

    # Esta consulta es idéntica conceptualmente al: SELECT * FROM documents ORDER BY embedding <=> vec LIMIT k; de pgvector
    cursor.execute("""
        SELECT id, COSINE_DISTANCE(?, embedding) as distancia, text, doc_id
        FROM documents
        ORDER BY distancia ASC
        LIMIT ?
    """, (query_json, k))

    rows = cursor.fetchall()
    results = []
    for r in rows:
        results.append((r[0], r[1], r[2], {"doc_id": r[3]}))
    return results

print("\n--- Resultados de Búsqueda simulando pgvector (SQL) ---")
sql_res = pgvector_search(query_embedding, k=5)
for res in sql_res:
    print(f"ID (PK): {res[0]} | Distancia Calculada por SQL: {res[1]:.4f}")
    print(f"Texto: {res[2][:120]}...\n")

conn.close()


--- Resultados de Búsqueda simulando pgvector (SQL) ---
ID (PK): 10176 | Distancia Calculada por SQL: 0.1297
Texto: Battery tester A battery tester is an electronic device intended for testing the state of an electric battery, going fro...

ID (PK): 1 | Distancia Calculada por SQL: 0.1382
Texto: Battery indicator A battery indicator (also known as a battery gauge) is a device which gives information about a batter...

ID (PK): 10177 | Distancia Calculada por SQL: 0.1599
Texto: ing procedure, according to the type of battery being tested, such as the â€œ421â€ test for lead-acid vehicle batteries...

ID (PK): 37406 | Distancia Calculada por SQL: 0.1609
Texto: ils. One was connected via a series resistor to the battery supply. The second was connected to the same battery supply ...

ID (PK): 71872 | Distancia Calculada por SQL: 0.1614
Texto: is achieved. Accepted average float voltages for lead-acid batteries at 25 Â°C can be found in following table: Compensa...



**¿Qué tan “explicable” te parece esta aproximación vs las otras?**  
Es la más transparente de todas por lejos. Cualquier desarrollador de software entiende un SELECT ... FROM ... ORDER BY distancia LIMIT k. No hay cajas negras de librerías externas ni APIs raras; es pura álgebra lineal aplicada sobre un motor relacional que ya conocemos.

**¿Qué ventajas ofrece el mundo SQL (JOIN, filtros, agregaciones)?**   
La ventaja es masiva. Te permite cruzar tus datos vectoriales semánticos con transacciones operacionales usando un JOIN tradicional instantáneamente (ej. buscar reviews similares de películas, pero filtrando únicamente los usuarios que tienen una membresía activa y compraron en la última semana). Todo en una sola consulta ACID robusta.

**¿Qué limitaciones esperas en escalabilidad frente a bases vectoriales dedicadas?**  
SQL no fue diseñado originalmente para buscar en espacios multidimensionales de alta densidad. Hacer un escaneo completo (Full Table Scan) calculando distancias vectoriales para millones de filas en cada query bloquea la base de datos por completo. Aunque pgvector tiene índices como IVFFlat o HNSW, las bases dedicadas (como Qdrant o Milvus) distribuyen el cómputo y manejan la memoria de una forma infinitamente más eficiente para cargas masivas de Big Data.